In [ ]:
import os
import sys
import pandas as pd
import matplotlib.pyplot as plt
from natsort import natsorted
from tqdm import tqdm
import requests
import yaml
from Bio import SeqIO, Entrez
from io import StringIO

sys.path.append("..")

from dataset.bactero_set import BacteriaDataset, TAXO_LEVELS


In [9]:
!pwd

/home/hcourtei/Projects/MicroTaxo/codes/wisp/wisp_light/notebook


In [29]:
completed_df  = pd.read_csv('assembly_summary_Complete_Genome.csv', sep='\t')
completed_df

,assembly_accession,refseq_category,ftp_path,Downloaded
0,GCF_900128725.1,na,https://ftp.ncbi.nlm.nih.gov/genomes/all/GCF/9...,False
1,GCF_003044255.1,na,https://ftp.ncbi.nlm.nih.gov/genomes/all/GCF/0...,False
2,GCF_009730575.1,na,https://ftp.ncbi.nlm.nih.gov/genomes/all/GCF/0...,False
3,GCF_016406305.1,na,https://ftp.ncbi.nlm.nih.gov/genomes/all/GCF/0...,False
4,GCF_016406325.1,reference genome,https://ftp.ncbi.nlm.nih.gov/genomes/all/GCF/0...,False
...,...,...,...,...
44969,GCF_044998965.1,na,https://ftp.ncbi.nlm.nih.gov/genomes/all/GCF/0...,False
44970,GCF_041154365.1,na,https://ftp.ncbi.nlm.nih.gov/genomes/all/GCF/0...,False
44971,GCF_044788715.1,na,https://ftp.ncbi.nlm.nih.gov/genomes/all/GCF/0...,False
44972,GCF_045037995.1,na,https://ftp.ncbi.nlm.nih.gov/genomes/all/GCF/0...,False


In [30]:
def add_accesion_from_file(df_in, output_dir):
    # Ajouter une nouvelle colonne ou mettre à jour la colonne Downloaded
    df = df_in.copy()
    for index, row in tqdm(df.iterrows()):
        # Extraire l'URL FTP et créer le chemin du fichier attendu
        ftp_path = row['ftp_path']
        end_url_file = ftp_path[8:].split('/')[-1]  # On enlève 'https://' et on prend la dernière partie de l'URL
        file_path = os.path.join(output_dir, f"{end_url_file}_genomic.fna.gz")
        unzip_file_path = file_path.removesuffix(".gz")


        # Vérifier si le fichier .fna.gz existe
        if os.path.isfile(unzip_file_path):
            df.at[index, 'file'] = os.path.basename(unzip_file_path)
            df.at[index, 'Downloaded'] = True
            with open(unzip_file_path, "r", encoding='utf-8') as reader:
                # first_line = reader.readline().strip()
                first_line = reader.readline()
                accession = first_line.split('.')[0][1:]  # Extraction de l'accession
                df.at[index, 'accession']  = accession
        else:
            df.at[index, 'Downloaded'] = False
    return df

In [31]:
output_dir = '/home/hcourtei/Projects/MicroTaxo/codes/wisp/wisp_light/import_dataset/refseq/out_refseq'
#output_dir = '/projects/microtaxo/data/refseq2'

df_with_accession = add_accesion_from_file(completed_df, output_dir)
print(df_with_accession[380:420].drop(columns=['ftp_path']).to_markdown())
df_with_accession.to_csv('Complete_Genome_with_accession.csv', sep='\t')


44974it [00:01, 35394.98it/s]


|     | assembly_accession   | refseq_category   | Downloaded   | file                                     | accession   |
|----:|:---------------------|:------------------|:-------------|:-----------------------------------------|:------------|
| 380 | GCF_017352015.1      | na                | True         | GCF_017352015.1_ASM1735201v1_genomic.fna | NZ_CP071460 |
| 381 | GCF_017356685.1      | na                | True         | GCF_017356685.1_ASM1735668v1_genomic.fna | NZ_CP071519 |
| 382 | GCF_017356745.1      | na                | True         | GCF_017356745.1_ASM1735674v1_genomic.fna | NZ_CP071574 |
| 383 | GCF_017356765.1      | na                | True         | GCF_017356765.1_ASM1735676v1_genomic.fna | NZ_CP071575 |
| 384 | GCF_017356785.1      | na                | True         | GCF_017356785.1_ASM1735678v1_genomic.fna | NZ_CP071576 |
| 385 | GCF_017356805.1      | na                | True         | GCF_017356805.1_ASM1735680v1_genomic.fna | NZ_CP071579 |
| 386 | GCF_0173

In [ ]:
# https://biopython.org/docs/1.76/api/Bio.Entrez.html
def get_batch_taxo_in_df(df_in, batch_size):
    df = df_in.copy()
    Entrez.email = "hermann.courteille@inria.fr"
    Entrez.api_key =  "b55513ab1634ec527ccf1ec084f3b1c78108"
    Entrez.max_tries = 5
    Entrez.sleep_between_tries = 15

    res = []
    
    with Entrez.efetch(db="nucleotide", id=batch, rettype="gb", retmode="text") as taxo_handle:
        records = SeqIO.parse(taxo_handle, 'genbank')

        for idx , record in tqdm(enumerate(records)):
            taxonomy = record.annotations.get('taxonomy', [])
            organism = record.annotations.get('organism', "Unknown Organism")
            
            if len(taxonomy)>7 or not taxonomy[4].endswith('ales'):
                print("taxo: ", taxonomy, "\norga :", organism)
            order = next((e for e in taxonomy if e.endswith('ales')), None)
            
            if order:
                regne, phylum = taxonomy[0], taxonomy[1]
                group = taxonomy[2] if len(taxonomy) > 2 and not taxonomy[2].endswith('ales') else taxonomy[1]
                
            family, specie = organism.split(' ')[:2]
            others = organism.split(' ')[2:]
            
            res.append((regne, phylum, group, order, family, specie, others))

    return df


In [23]:
batch  = list(df_with_accession['accession'][:20]) #['NZ_LT667500', 'NZ_CP028435', 'NZ_CP046329', 'NZ_CP066369', 'NZ_CP066370'] # 
res = get_batch_taxo_in_df(batch)

9it [00:19,  1.86s/it]

taxo:  ['Bacteria', 'Pseudomonadati', 'Myxococcota', 'Myxococcia', 'Myxococcales', 'Cystobacterineae', 'Myxococcaceae', 'Myxococcus'] 
orga : Myxococcus xanthus


10it [00:21,  1.91s/it]

taxo:  ['Bacteria', 'Pseudomonadati', 'Myxococcota', 'Myxococcia', 'Myxococcales', 'Cystobacterineae', 'Myxococcaceae', 'Myxococcus'] 
orga : Myxococcus xanthus
taxo:  ['Bacteria', 'Pseudomonadati', 'Myxococcota', 'Myxococcia', 'Myxococcales', 'Cystobacterineae', 'Myxococcaceae', 'Myxococcus'] 
orga : Myxococcus xanthus


12it [00:23,  1.44s/it]

...

taxo:  ['Bacteria', 'Pseudomonadati', 'Myxococcota', 'Myxococcia', 'Myxococcales', 'Cystobacterineae', 'Myxococcaceae', 'Myxococcus'] 
orga : Myxococcus xanthus


18it [00:35,  1.95s/it]

taxo:  ['Bacteria', 'Pseudomonadati', 'Myxococcota', 'Myxococcia', 'Myxococcales', 'Cystobacterineae', 'Myxococcaceae', 'Corallococcus'] 
orga : Corallococcus macrosporus


20it [00:38,  1.95s/it]

taxo:  ['Bacteria', 'Pseudomonadati', 'Myxococcota', 'Myxococcia', 'Myxococcales', 'Cystobacterineae', 'Archangiaceae', 'Cystobacter'] 
orga : Cystobacter fuscus
taxo:  ['Bacteria', 'Pseudomonadati', 'Myxococcota', 'Myxococcia', 'Myxococcales', 'Cystobacterineae', 'Archangiaceae', 'Cystobacter'] 
orga : Cystobacter fuscus
